# 02. 데이터 분할과 공용 입력 파일 만들기

입력: `data/processed/phiusiil_clean.csv` (01 노트북 결과)

출력 (팀 전체가 공유)
| 파일 | 내용 |
|---|---|
| `data/processed/dataset.csv` | 정리된 데이터 + `uid` + `url_stripped` |
| `data/splits/splits.csv` | `uid`별 분할 (`split_random`, `split_domain`, `split_domain_public`)과 train 내부 5-fold 번호 |
| `data/splits/features.json` | 시나리오별 입력 열 목록 (S2는 **초안**) |

사용법: `from src.data import load_split` → `load_split(scenario="S2", split="domain")`

원칙
- 분할 시드는 `SEED = 42` 하나로 고정한다. 반복 실험은 모델 시드만 바꾼다 (같은 Test에서 McNemar 검정을 하기 위해).
- 비율은 train/val/test = 70/15/15.
- 도메인 분할: 그룹을 유지하며 클래스 비율을 맞춘 20조각을 만들어 14/3/3조각씩 묶는다.

In [1]:
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, train_test_split

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

PROC = ROOT / "data" / "processed"
SPLIT_DIR = ROOT / "data" / "splits"
FIG = ROOT / "reports" / "figures"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_CHUNKS = 20                       # 도메인 분할용 조각 수 (5%씩)
TRAIN_CHUNKS, VAL_CHUNKS = 14, 3    # 나머지 3조각이 test
N_CV = 5
PARTS = ["train", "val", "test"]

## 1. 데이터 불러오기, `uid`와 `url_stripped` 추가

In [2]:
df = pd.read_csv(PROC / "phiusiil_clean.csv", encoding="utf-8")
assert df["URL"].is_unique
df.insert(0, "uid", np.arange(len(df)))

STRIP = re.compile(r"^(?:https?://)?(?:www\.)?", flags=re.IGNORECASE)
def strip_url(u: str) -> str:
    """scheme, 맨 앞 www., 끝 슬래시를 지운다. 정상/피싱 수집 방식의 흔적을 없애기 위한 S3 입력."""
    s = STRIP.sub("", u.strip())
    return s.rstrip("/")

df.insert(df.columns.get_loc("URL") + 1, "url_stripped", df["URL"].map(strip_url))
print(df.shape)
df[["uid", "URL", "url_stripped", "etld1", "etld1_public", "is_phishing"]].sample(8, random_state=0)

(235370, 60)


,uid,URL,url_stripped,etld1,etld1_public,is_phishing
34385,34385,https://www.saic.edu,saic.edu,saic.edu,saic.edu,0
140967,140967,https://www.fr12.nl,fr12.nl,fr12.nl,fr12.nl,0
143071,143071,http://www.rakoten-cacd.xyfwczq.cn/,rakoten-cacd.xyfwczq.cn,xyfwczq.cn,xyfwczq.cn,1
230225,230225,http://www.xite.cc,xite.cc,xite.cc,xite.cc,1
74048,74048,https://long-mountain-ad58.exkuqtj.workers.dev/,long-mountain-ad58.exkuqtj.workers.dev,exkuqtj.workers.dev,workers.dev,1
83860,83860,http://mail.deliverylifesupport.com/public/lpe...,mail.deliverylifesupport.com/public/lpe8nhfy6o...,deliverylifesupport.com,deliverylifesupport.com,1
144500,144500,https://www.bitsdujour.com,bitsdujour.com,bitsdujour.com,bitsdujour.com,0
190852,190852,http://www.stmarouns.nsw.edu.au,stmarouns.nsw.edu.au,stmarouns.nsw.edu.au,stmarouns.nsw.edu.au,1


In [3]:
# 정리 후 문자열이 같아지는 URL (예: http://www.a.com 과 https://a.com/) 이 클래스를 넘나드는지 확인
g = df.groupby("url_stripped")["is_phishing"].agg(["size", "nunique"])
print(f"url_stripped 중복 그룹 {(g['size'] > 1).sum():,}, 그 중 라벨 충돌 {(g['nunique'] > 1).sum():,}")
conf = g[g["nunique"] > 1].index
df[df["url_stripped"].isin(conf)].sort_values("url_stripped")[["URL", "url_stripped", "is_phishing"]].head(10)

url_stripped 중복 그룹 3,276, 그 중 라벨 충돌 43


,URL,url_stripped,is_phishing
18288,http://www.1x.com,1x.com,1
70235,https://www.1x.com,1x.com,0
72139,http://www.adgf.am,adgf.am,1
74660,https://www.adgf.am,adgf.am,0
214693,http://www.airlinemeals.net,airlinemeals.net,1
28124,https://www.airlinemeals.net,airlinemeals.net,0
135330,https://www.alphauniforms.ae,alphauniforms.ae,0
64469,http://www.alphauniforms.ae,alphauniforms.ae,1
209034,http://www.amna.gr,amna.gr,1
49944,https://www.amna.gr,amna.gr,0


> `url_stripped` 기준 충돌 행은 지우지 않고 그대로 둔다. 원본 URL은 서로 다르기 때문이다.
> S3에서 `url_stripped`를 쓸 때 이 행들은 입력이 같은데 라벨이 다른 샘플이 된다. 규모가 작으면 무시하고, 크면 팀 논의로 처리 방식을 정한다.

## 2. 무작위 분할 (클래스 비율 유지)

In [4]:
y = df["is_phishing"].to_numpy()
idx = np.arange(len(df))
tr, rest = train_test_split(idx, test_size=0.30, stratify=y, random_state=SEED)
va, te = train_test_split(rest, test_size=0.50, stratify=y[rest], random_state=SEED)

split_random = np.empty(len(df), dtype=object)
split_random[tr], split_random[va], split_random[te] = "train", "val", "test"
df["split_random"] = split_random

## 3. 도메인 단위 분할
그룹 키 두 가지를 모두 만든다.
- `split_domain`: `etld1` (private suffix 포함, 소유자 단위, 예: `abc.web.app`)
- `split_domain_public`: `etld1_public` (플랫폼 단위, 예: `web.app`)

> sklearn `StratifiedGroupKFold`는 그룹이 약 20만 개일 때 10분 넘게 걸려서, 같은 목적의 빠른 구현 `src/split_utils.stratified_group_chunks`를 쓴다.
> 큰 그룹부터 가장 덜 찬 조각에 넣고, URL 1개짜리 그룹으로 클래스 비율을 맞춘다. 같은 그룹은 항상 같은 조각에 들어간다.

In [5]:
from src.split_utils import stratified_group_chunks

def group_split(groups: pd.Series, y: np.ndarray, seed: int = SEED) -> np.ndarray:
    chunk = stratified_group_chunks(groups, y, N_CHUNKS, seed)
    out = np.where(chunk < TRAIN_CHUNKS, "train",
                   np.where(chunk < TRAIN_CHUNKS + VAL_CHUNKS, "val", "test"))
    return out.astype(object)

df["split_domain"] = group_split(df["etld1"], y)
df["split_domain_public"] = group_split(df["etld1_public"], y)

## 4. train 내부 5-fold 번호 (하이퍼파라미터 튜닝용)
- 무작위 분할: `StratifiedKFold`
- 도메인 분할: `stratified_group_chunks` (같은 도메인이 같은 fold)

train이 아닌 행은 -1.

In [6]:
def cv_folds(split_col: str, groups: pd.Series | None) -> np.ndarray:
    fold = np.full(len(df), -1, dtype=int)
    tr_idx = np.where(df[split_col] == "train")[0]
    y_tr = y[tr_idx]
    if groups is None:
        it = StratifiedKFold(N_CV, shuffle=True, random_state=SEED).split(tr_idx, y_tr)
    else:
        fold[tr_idx] = stratified_group_chunks(groups.iloc[tr_idx], y_tr, N_CV, SEED)
        return fold
    for k, (_, v) in enumerate(it):
        fold[tr_idx[v]] = k
    return fold

df["cv_fold_random"] = cv_folds("split_random", None)
df["cv_fold_domain"] = cv_folds("split_domain", df["etld1"])
df["cv_fold_domain_public"] = cv_folds("split_domain_public", df["etld1_public"])

## 5. 분할 검증
### 5-1. 크기와 클래스 비율

In [7]:
SPLITS = {"random": "split_random", "domain": "split_domain", "domain_public": "split_domain_public"}
rows = []
for name, col in SPLITS.items():
    for p in PARTS:
        m = df[col] == p
        rows.append({"split": name, "part": p, "n": int(m.sum()), "ratio": m.mean(),
                     "phishing_rate": df.loc[m, "is_phishing"].mean()})
size_tbl = pd.DataFrame(rows).set_index(["split", "part"]).round(4)
size_tbl

n   ratio  phishing_rate
split         part                                
random        train  164759  0.7000         0.4271
              val     35305  0.1500         0.4271
              test    35306  0.1500         0.4271
domain        train  164762  0.7000         0.4271
              val     35304  0.1500         0.4271
              test    35304  0.1500         0.4271
domain_public train  164418  0.6986         0.4259
              val     35858  0.1523         0.4359
              test    35094  0.1491         0.4237

### 5-2. 도메인 겹침
도메인 분할은 겹침이 0이어야 한다. 무작위 분할은 test 도메인이 train에 얼마나 들어가 있는지 보여준다 (도메인 중복으로 성능이 부풀 수 있는 정도).

In [8]:
def overlap(col: str, key: str) -> dict:
    s = {p: set(df.loc[df[col] == p, key]) for p in PARTS}
    te_rows = df[col] == "test"
    return {
        "train∩val 도메인": len(s["train"] & s["val"]),
        "train∩test 도메인": len(s["train"] & s["test"]),
        "val∩test 도메인": len(s["val"] & s["test"]),
        "train에 도메인이 있는 test 행 비율": round(df.loc[te_rows, key].isin(s["train"]).mean(), 4),
    }

ov = pd.DataFrame({
    f"{name} / {key}": overlap(col, key)
    for name, col in SPLITS.items() for key in ["etld1", "etld1_public"]
}).T
ov

,train∩val 도메인,train∩test 도메인,val∩test 도메인,train에 도메인이 있는 test 행 비율
random / etld1,2576.0,2581.0,1184.0,0.1778
random / etld1_public,2231.0,2211.0,1092.0,0.2692
domain / etld1,0.0,0.0,0.0,0.0000
domain / etld1_public,97.0,94.0,72.0,0.1256
domain_public / etld1,0.0,0.0,0.0,0.0000
domain_public / etld1_public,0.0,0.0,0.0,0.0000


In [9]:
# cv fold 안에서도 도메인이 섞이지 않는지
for split, key in [("domain", "etld1"), ("domain_public", "etld1_public")]:
    tr = df[df[f"split_{split}"] == "train"]
    n_multi = tr.groupby(key)[f"cv_fold_{split}"].nunique().gt(1).sum()
    sizes = tr[f"cv_fold_{split}"].value_counts().sort_index().tolist()
    rates = tr.groupby(f"cv_fold_{split}")["is_phishing"].mean().round(4).tolist()
    print(f"[{split}] 여러 fold에 걸친 도메인: {n_multi} | fold 크기 {sizes} | 피싱 비율 {rates}")
tr = df[df["split_random"] == "train"]
print("[random] fold 크기", tr["cv_fold_random"].value_counts().sort_index().tolist())

[domain] 여러 fold에 걸친 도메인: 0 | fold 크기 [32953, 32953, 32953, 32952, 32951] | 피싱 비율 [0.4271, 0.4271, 0.4271, 0.4271, 0.4271]


[domain_public] 여러 fold에 걸친 도메인: 0 | fold 크기 [32884, 32884, 32884, 32883, 32883] | 피싱 비율 [0.4259, 0.4259, 0.4259, 0.4259, 0.4259]
[random] fold 크기 [32952, 32952, 32952, 32952, 32951]


### 5-3. 대형 그룹이 어디로 갔는지

In [10]:
big = ["web.app", "firebaseapp.com", "repl.co", "weeblysite.com", "google.com", "ipfs.io", "github.io"]
pd.DataFrame({
    g: df[df["etld1_public"] == g]["split_domain_public"].value_counts().reindex(PARTS, fill_value=0)
    for g in big
}).T.assign(total=lambda t: t.sum(axis=1))

split_domain_public,train,val,test,total
web.app,0,5718,0,5718
firebaseapp.com,5546,0,0,5546
repl.co,3746,0,0,3746
weeblysite.com,3079,0,0,3079
google.com,782,0,0,782
ipfs.io,1552,0,0,1552
github.io,400,0,0,400


### 5-4. 분할별 URL 형태 분포
01 노트북 7-2절의 형태 편향이 분할마다 비슷하게 퍼졌는지 확인한다. 한 분할에 특정 형태가 몰리면 분할 간 성능 차이가 형태 차이 때문일 수 있다.

In [11]:
from urllib.parse import urlsplit

def url_form(u: str) -> str:
    p = urlsplit(u)
    tail = "쿼리" if p.query else "경로" if p.path not in ("", "/") else "끝 슬래시만" if p.path == "/" else "없음"
    www = "www" if u.lower().split("://", 1)[-1].startswith("www.") else "no-www"
    return f"{p.scheme.lower()}|{www}|{tail}"

form = df["URL"].map(url_form)
top = form[df["is_phishing"] == 1].value_counts().head(6).index
ph = df["is_phishing"] == 1
form_tbl = pd.concat({
    name: pd.crosstab(form[ph].where(form[ph].isin(top), "기타"), df.loc[ph, col], normalize="columns")[PARTS]
    for name, col in SPLITS.items()
}, axis=1).mul(100).round(1).rename_axis(index="피싱 URL 형태", columns=["split", "part"])
form_tbl

split               random             domain             domain_public        \
part                 train   val  test  train   val  test         train   val   
피싱 URL 형태                                                                       
https|no-www|경로       15.3  15.2  15.3   15.0  16.9  14.8          14.9  15.4   
https|no-www|끝 슬래시만   25.6  25.7  25.8   26.3  21.8  26.6          25.1  42.0   
https|no-www|쿼리        3.6   3.8   3.7    3.4   5.4   2.8           3.9   2.3   
http|no-www|경로         4.5   4.6   4.3    4.6   3.8   4.5           4.7   2.2   
http|no-www|끝 슬래시만     4.2   4.4   4.2    4.3   3.9   4.2           4.2   2.9   
http|www|없음           36.3  35.5  36.4   35.9  37.7  36.0          37.0  27.4   
기타                    10.5  10.7  10.3   10.4  10.5  11.1          10.2   7.8   

split                      
part                 test  
피싱 URL 형태                  
https|no-www|경로      17.1  
https|no-www|끝 슬래시만  11.2  
https|no-www|쿼리       3.6  
http|no-www|경로        5.9  
http|no-www|끝 슬래시만    5.6  
http|www|없음          41.7  
기타                   14.9

## 6. 피처 목록 (`features.json`)

In [12]:
TEXT_COLS = ["URL", "url_stripped", "Domain", "TLD", "Title", "etld1", "etld1_public"]
META_COLS = ["uid", "label", "is_phishing"]
S1 = [c for c in df.columns if c not in TEXT_COLS + META_COLS and not c.startswith(("split_", "cv_fold_"))]

# S2 초안: 01 노트북 8절의 기준 A, B 후보. 11/18 팀 회의에서 확정한다.
S2_REMOVE_DRAFT = ["URLSimilarityIndex", "TLDLegitimateProb", "URLCharProb",
                   "LineOfCode", "LargestLineLength", "HasSocialNet"]
S2_REVIEW = ["CharContinuationRate", "HasCopyrightInfo", "HasDescription",
             "NoOfExternalRef", "NoOfSelfRef", "IsHTTPS"]
assert set(S2_REMOVE_DRAFT + S2_REVIEW) <= set(S1)
S2 = [c for c in S1 if c not in S2_REMOVE_DRAFT]

features = {
    "S1": S1,
    "S2": S2,
    "S3": ["URL", "url_stripped"],
    "S2_removed": S2_REMOVE_DRAFT,
    "S2_review": S2_REVIEW,
    "S2_status": "draft (2026-11-18 팀 확정 전)",
    "excluded_always": ["FILENAME", "Domain", "TLD", "Title"],
    "label": "is_phishing (1=피싱, 0=정상)",
}
print(f"S1 {len(S1)}개, S2 {len(S2)}개 (초안, {len(S2_REMOVE_DRAFT)}개 제거, {len(S2_REVIEW)}개 검토 대기)")

S1 50개, S2 44개 (초안, 6개 제거, 6개 검토 대기)


## 7. 저장

In [13]:
split_cols = ["uid"] + list(SPLITS.values()) + ["cv_fold_random", "cv_fold_domain", "cv_fold_domain_public"]
data_cols = [c for c in df.columns if c not in split_cols[1:]]

df[data_cols].to_csv(PROC / "dataset.csv", index=False, encoding="utf-8")
df[split_cols].to_csv(SPLIT_DIR / "splits.csv", index=False)
with open(SPLIT_DIR / "features.json", "w", encoding="utf-8") as f:
    json.dump(features, f, ensure_ascii=False, indent=2)
size_tbl.to_csv(SPLIT_DIR / "split_summary.csv", encoding="utf-8-sig")

for p in [PROC / "dataset.csv", SPLIT_DIR / "splits.csv", SPLIT_DIR / "features.json", SPLIT_DIR / "split_summary.csv"]:
    print(f"{p.relative_to(ROOT)}  {p.stat().st_size/1e6:.1f} MB")

data\processed\dataset.csv  72.0 MB
data\splits\splits.csv  7.3 MB
data\splits\features.json  0.0 MB
data\splits\split_summary.csv  0.0 MB


## 8. 불러오기 확인 (`src/data.py`)

In [14]:
import importlib
import src.data as data
importlib.reload(data)

full = data.load_dataset()
for sc, sp in [("S1", "random"), ("S2", "domain"), ("S3", "domain")]:
    d = data.load_split(sc, sp, df=full)
    shape = d.X_train.shape
    print(f"{sc} / {sp:14s} X_train {shape}  cv_fold {sorted(map(int, d.cv_fold.unique()))}")
    print(d.summary().T.to_string(), "\n")

S1 / random         X_train (164759, 50)  cv_fold [0, 1, 2, 3, 4]
part                 train         val        test
n              164759.0000  35305.0000  35306.0000
phishing_rate       0.4271      0.4271      0.4271 



S2 / domain         X_train (164762, 44)  cv_fold [0, 1, 2, 3, 4]
part                 train         val        test
n              164762.0000  35304.0000  35304.0000
phishing_rate       0.4271      0.4271      0.4271 



S3 / domain         X_train (164762,)  cv_fold [0, 1, 2, 3, 4]
part                 train         val        test
n              164762.0000  35304.0000  35304.0000
phishing_rate       0.4271      0.4271      0.4271 



In [15]:
# S3 문자 인코딩: 사전과 최대 길이는 train으로만 정한다
d3 = data.load_split("S3", "domain", url_col="url_stripped", df=full)
vocab = data.build_vocab(d3.X_train)
max_len = data.choose_max_len(d3.X_train, q=0.99)
X_tr_ids = data.encode_chars(d3.X_train, vocab, max_len)
X_te_ids = data.encode_chars(d3.X_test, vocab, max_len)
print(f"vocab {len(vocab)}자 (+PAD, UNK) | max_len {max_len} | train {X_tr_ids.shape} | test {X_te_ids.shape}")
print(f"test에서 UNK 비율: {(X_te_ids == data.UNK).sum() / (X_te_ids != data.PAD).sum():.5f}")
print(f"max_len보다 긴 train URL 비율: {(d3.X_train.str.len() > max_len).mean():.4f}")
print("원본 URL 기준 max_len:", data.choose_max_len(data.load_split('S3', 'domain', df=full).X_train))

vocab 65자 (+PAD, UNK) | max_len 121 | train (164762, 121) | test (35304, 121)
test에서 UNK 비율: 0.00000
max_len보다 긴 train URL 비율: 0.0100


원본 URL 기준 max_len: 130


## 9. 팀원용 사용법

```python
import sys; sys.path.insert(0, "..")          # notebooks/ 안에서 실행할 때
from src.data import load_split, build_vocab, choose_max_len, encode_chars

# 트랙 A (정형 피처): S1 또는 S2, 분할은 "random" / "domain" / "domain_public"
d = load_split("S2", "domain")
# GroupKFold 튜닝: d.cv_fold 를 PredefinedSplit 에 넘기거나, d.groups 를 GroupKFold 에 넘긴다
from sklearn.model_selection import PredefinedSplit
cv = PredefinedSplit(d.cv_fold)

# 트랙 B (URL 문자열): S3, url_col="URL" 또는 "url_stripped"
d = load_split("S3", "domain", url_col="url_stripped")
vocab = build_vocab(d.X_train); L = choose_max_len(d.X_train)
X_train = encode_chars(d.X_train, vocab, L)
```
- `y`는 항상 `is_phishing` (피싱=1). 평가 지표의 양성 클래스와 같다.
- 표준화는 `Pipeline([("scaler", StandardScaler()), ("model", ...)])`처럼 모델 쪽에서 한다.
- 예측 결과를 저장할 때 `d.uid["test"]`를 함께 저장하면 모델 간 McNemar 검정과 오류 분석에서 행을 맞출 수 있다.